# Video HLS Generator
Supports a **single video URL** or a **Google Drive folder** with multiple videos.

> **Tip:** Go to **Runtime → Change runtime type → T4 GPU** for full GPU pipeline.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q yt-dlp gdown

import subprocess

caps = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True)
USE_GPU = "h264_nvenc" in caps.stdout

if USE_GPU:
    print("✓ GPU detected — CUDA decode + NVENC encode (fast)")
else:
    print("⚠ No GPU found — using libx264 (CPU). Enable T4 in Runtime settings.")

In [ ]:
# ── Configuration — edit these ────────────────────────────────────────────

# Option A: Google Drive shared folder URL (processes ALL videos inside)
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1ojVwHDmIygJFNB4x74jPyS_W9VwTCEjq?usp=sharing"

# Option B: Single direct video URL (YouTube, direct .mp4 link, etc.)
# Set DRIVE_FOLDER_URL = "" and fill this instead
SINGLE_VIDEO_URL = ""

SELECTED_RESOLUTIONS = ["1080p", "720p", "480p", "360p"]  # remove any you don't need

CPU_THREADS = 2  # only used when GPU is not available

In [ ]:
# ── List Drive folder & select files to download ──────────────────────────────
import re, os, shutil, gdown
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".flv", ".m4v"}
VIDEO_MIME_TYPES  = {"video/mp4", "video/quicktime", "video/x-msvideo",
                     "video/x-matroska", "video/webm", "video/x-flv", "video/x-m4v"}

file_checkboxes = {}   # {file_meta_dict: Checkbox}

if DRIVE_FOLDER_URL:
    # Authenticate with Google (one-click in Colab)
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build

    drive_service = build("drive", "v3", cache_discovery=False)

    # Extract folder ID from URL
    match = re.search(r"/folders/([a-zA-Z0-9_-]+)", DRIVE_FOLDER_URL)
    if not match:
        raise ValueError("Could not extract folder ID from DRIVE_FOLDER_URL")
    folder_id = match.group(1)

    # List all video files in the folder
    results = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, mimeType, size)",
        pageSize=100
    ).execute()

    all_files = results.get("files", [])
    video_files_meta = [
        f for f in all_files
        if f.get("mimeType", "") in VIDEO_MIME_TYPES
        or Path(f.get("name", "")).suffix.lower() in VIDEO_EXTENSIONS
    ]
    video_files_meta.sort(key=lambda f: f["name"])

    if not video_files_meta:
        raise RuntimeError("No video files found in the Drive folder.")

    # Build checkboxes
    select_all_btn = widgets.ToggleButton(
        value=True, description="Select All", button_style="info",
        layout=widgets.Layout(width="120px", margin="0 0 10px 0")
    )
    for meta in video_files_meta:
        size_mb = int(meta.get("size", 0)) / 1024 / 1024
        label = f"{meta['name']}  ({size_mb:.1f} MB)" if size_mb > 0 else meta["name"]
        cb = widgets.Checkbox(value=True, description=label,
                              layout=widgets.Layout(width="550px"))
        file_checkboxes[meta["id"]] = {"cb": cb, "name": meta["name"], "id": meta["id"]}

    def toggle_all(change):
        for v in file_checkboxes.values():
            v["cb"].value = change["new"]
    select_all_btn.observe(toggle_all, names="value")

    print(f"Found {len(video_files_meta)} video(s) in Drive folder.")
    print("Check the files you want to download, then run the next cell.\n")
    display(select_all_btn)
    display(widgets.VBox([v["cb"] for v in file_checkboxes.values()]))

else:
    print("Single URL mode — skip to the next cell.")

In [ ]:
# ── Download selected files ───────────────────────────────────────────────────
os.makedirs("input", exist_ok=True)
video_files = []

if DRIVE_FOLDER_URL:
    selected = [v for v in file_checkboxes.values() if v["cb"].value]
    if not selected:
        raise ValueError("No files selected. Go back and check at least one box.")

    print(f"Downloading {len(selected)} file(s)...\n")
    for item in selected:
        out_path = os.path.join("input", item["name"])
        print(f"  ↓ {item['name']}")
        gdown.download(id=item["id"], output=out_path, quiet=True)
        video_files.append(out_path)

elif SINGLE_VIDEO_URL:
    print(f"Downloading: {SINGLE_VIDEO_URL}")
    result = subprocess.run(
        ["yt-dlp", "--no-playlist", "-o", "input/%(title)s.%(ext)s", SINGLE_VIDEO_URL],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        filename = SINGLE_VIDEO_URL.split("/")[-1].split("?")[0] or "video.mp4"
        subprocess.run(["wget", "-q", "-O", f"input/{filename}", SINGLE_VIDEO_URL])
    video_files = sorted([
        str(p) for p in Path("input").iterdir()
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    ])
else:
    raise ValueError("Set either DRIVE_FOLDER_URL or SINGLE_VIDEO_URL in the config cell.")

print(f"\n✓ Ready to process {len(video_files)} file(s):")
for v in video_files:
    print(f"  • {Path(v).name}")

In [ ]:
# ── HLS Generation ────────────────────────────────────────────────────────────

RESOLUTIONS = {
    "1080p": {"width": 1920, "height": 1080, "bitrate": "5000k", "audio": "192k"},
    "720p":  {"width": 1280, "height": 720,  "bitrate": "2800k", "audio": "128k"},
    "480p":  {"width": 854,  "height": 480,  "bitrate": "1400k", "audio": "128k"},
    "360p":  {"width": 640,  "height": 360,  "bitrate": "800k",  "audio": "96k"},
}

MASTER_BANDWIDTHS = {
    "1080p": 5000000, "720p": 2800000, "480p": 1400000, "360p": 800000,
}

def _double_bitrate(bitrate: str) -> str:
    return str(int(bitrate.rstrip("k")) * 2) + "k"

print(f"Processing {len(video_files)} file(s):\n")

def encode_video(video_path, selected_res):
    stem = Path(video_path).stem
    job_dir = os.path.join("output", stem)
    os.makedirs(job_dir, exist_ok=True)

    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", video_path],
        capture_output=True, text=True
    )
    try:
        duration = float(probe.stdout.strip())
        print(f"  Duration: {duration:.1f}s")
    except Exception:
        duration = 0.0

    thumb_path = os.path.join(job_dir, "thumbnail.jpg")
    subprocess.run(
        ["ffmpeg", "-y", "-ss", "1", "-i", video_path,
         "-vframes", "1", "-q:v", "2", thumb_path],
        capture_output=True
    )

    for res in selected_res:
        os.makedirs(os.path.join(job_dir, res), exist_ok=True)

    n = len(selected_res)

    # ── filter_complex: decode once → split → scale each branch (CPU scale) ──
    # Note: always use software scale in filter_complex — mixing hwaccel_output_format
    # cuda with split/scale_cuda across multiple outputs is unreliable.
    # NVENC still handles encoding on the GPU for significant speedup.
    split_labels = "".join(f"[v{i}]" for i in range(n))
    filter_parts = [f"[0:v]split={n}{split_labels}"]
    for i, res in enumerate(selected_res):
        cfg = RESOLUTIONS[res]
        filter_parts.append(f"[v{i}]scale={cfg['width']}:{cfg['height']}[s{i}]")
    filter_complex = ";".join(filter_parts)

    # hwaccel for decode only (no hwaccel_output_format so frames stay accessible
    # to software filters like split/scale)
    hwaccel_flags = ["-hwaccel", "cuda"] if USE_GPU else []

    base_cmd = ["ffmpeg", "-y", *hwaccel_flags, "-i", video_path,
                "-filter_complex", filter_complex]
    if not USE_GPU:
        base_cmd += ["-threads", str(CPU_THREADS)]

    per_output = []
    for i, res in enumerate(selected_res):
        cfg = RESOLUTIONS[res]
        segment_pat = os.path.join(job_dir, res, "%03d.ts")
        playlist    = os.path.join(job_dir, res, "playlist.m3u8")

        if USE_GPU:
            # NVENC: GPU encode with VBR bitrate ceiling
            video_flags = ["-c:v", "h264_nvenc", "-preset", "p2", "-tune", "ll",
                           "-rc", "vbr", "-b:v", cfg["bitrate"], "-maxrate", cfg["bitrate"],
                           "-profile:v", "main"]
        else:
            # libx264: CRF with bitrate ceiling to control output size
            video_flags = ["-c:v", "libx264", "-crf", "23", "-preset", "medium",
                           "-profile:v", "main",
                           "-maxrate", cfg["bitrate"], "-bufsize", _double_bitrate(cfg["bitrate"])]

        per_output += [
            "-map", f"[s{i}]", "-map", "0:a?",
            *video_flags,
            "-c:a", "aac", "-b:a", cfg["audio"],
            "-hls_time", "6", "-hls_playlist_type", "vod",
            "-hls_segment_filename", segment_pat,
            playlist,
        ]

    cmd = base_cmd + per_output

    res_label = "+".join(selected_res)
    encoder_label = "NVENC" if USE_GPU else f"libx264 ({CPU_THREADS}t)"
    print(f"  Encoding [{res_label}] via {encoder_label} in one pass...", end="", flush=True)

    process = subprocess.Popen(cmd, stderr=subprocess.PIPE, stdout=subprocess.DEVNULL,
                               text=True, bufsize=1)

    stderr_lines = []
    last_pct = -1
    for line in process.stderr:
        stderr_lines.append(line)
        m = re.search(r"time=(\d+):(\d+):([\d.]+)", line)
        if m and duration:
            elapsed = int(m.group(1))*3600 + int(m.group(2))*60 + float(m.group(3))
            pct = min(int(elapsed / duration * 100), 100)
            if pct != last_pct and pct % 10 == 0:
                print(f" {pct}%", end="", flush=True)
                last_pct = pct

    process.wait()
    if process.returncode != 0:
        print(f"\n[FFmpeg failed — exit {process.returncode}]")
        # Print last 40 lines of stderr so the actual error is visible
        print("".join(stderr_lines[-40:]))
        raise RuntimeError(f"FFmpeg encoding failed (exit {process.returncode})")
    print(" ✓")

    master_path = os.path.join(job_dir, "master.m3u8")
    with open(master_path, "w") as f:
        f.write("#EXTM3U\n#EXT-X-VERSION:3\n")
        for res in selected_res:
            cfg = RESOLUTIONS[res]
            f.write(f"#EXT-X-STREAM-INF:BANDWIDTH={MASTER_BANDWIDTHS[res]},"
                    f"RESOLUTION={cfg['width']}x{cfg['height']}\n")
            f.write(f"{res}/playlist.m3u8\n")

    print(f"  ✓ master.m3u8 written")
    return job_dir


os.makedirs("output", exist_ok=True)
completed = []

for idx, video_path in enumerate(video_files):
    stem = Path(video_path).stem
    print(f"[{idx+1}/{len(video_files)}] Processing: {stem}")
    job_dir = encode_video(video_path, SELECTED_RESOLUTIONS)
    completed.append((stem, job_dir))
    print(f"  ✓ Done → {job_dir}\n")

print(f"=== All {len(completed)} video(s) processed ===")

In [ ]:
# ── Zip and download all outputs ──────────────────────────────────────────────
from google.colab import files

if len(completed) == 1:
    # Single video — zip just that folder
    stem, job_dir = completed[0]
    zip_path = shutil.make_archive(f"{stem}_hls", "zip", "output", stem)
    print(f"Zipped → {zip_path}")
    files.download(zip_path)
else:
    # Multiple videos — zip entire output folder as one archive
    zip_path = shutil.make_archive("hls_output_all", "zip", ".", "output")
    print(f"Zipped all → {zip_path}")
    files.download(zip_path)

In [ ]:
# ── Cleanup: delete input folder ─────────────────────────────────────────────
import shutil, os

if os.path.exists("input"):
    shutil.rmtree("input")
    print("Input folder deleted.")
else:
    print("Input folder does not exist.")

In [ ]:
# ── Cleanup: delete output folder ────────────────────────────────────────────
import shutil, os

if os.path.exists("output"):
    shutil.rmtree("output")
    print("Output folder deleted.")
else:
    print("Output folder does not exist.")